# Naive Bayes

In [141]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, GaussianNB
from sklearn.metrics import accuracy_score
import pickle

## Definición del problema

>### Análisis de sentimientos
>Los modelos Naive Bayes son muy útiles cuando queremos analizar sentimientos, clasificar textos en tópicos o recomendaciones, ya que las características de estos desafíos cumplen muy bien con los supuestos teóricos y metodológicos del modelo.

## Carga de datos

In [114]:
df = pd.read_csv('/workspaces/adamcn10-intro-ml/data/raw/playstore_reviews.csv')
df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


## Tratado de datos

### Análisis descriptivo

In [115]:
df.shape

(891, 3)

>Vemos que los datos se muestran en 891 filas y 3 columnas

In [116]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   package_name  891 non-null    object
 1   review        891 non-null    object
 2   polarity      891 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 21.0+ KB


>Estos datos se clasifican en package_name y review que son categóricas y polarity que es númerica pero actua como categórica y estas categorias se refieren a la aplicación que recibe el comentario, el comentario en si y si es positivo o no, respectivamente

In [117]:
df.nunique()

package_name     23
review          891
polarity          2
dtype: int64

>Vemos que se analizan 891 comentarios diferentes repartidos en 23 aplicaciones distintas.

### Limpieza de datos

#### Eliminamos duplicados

In [118]:
df.duplicated().sum()

np.int64(0)

>No hay datos duplicados

#### Eliminamos información irrelevante

>Como se dice en el ejercicio, solo nos interesa el comentario, no la aplicación por la que se haya escrito, así que eliminaremos la columna package_name

In [119]:
df.drop('package_name', axis=1, inplace=True)
df.shape

(891, 2)

>Los datos resultantes quedan solo con dos columnas, una predictora que es el comentario y una  objetivo que es la polaridad de este

In [120]:
df.head()

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0


>Nuevo aspecto del DataFrame

#### Tratamos los nulos

In [121]:
df.isnull().sum().sort_values(ascending=False)

review      0
polarity    0
dtype: int64

>Ya se podía ver en el info anterior, pero volvemos a ver que no tenemos valores nulos.

### Tratado de textos

In [122]:
df["review"] = df["review"].str.strip().str.lower()

### Split

In [123]:
X = df['review']
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=25)

## Modelado

In [124]:
vec_model = CountVectorizer(stop_words = "english")
X_train = vec_model.fit_transform(X_train).toarray()
X_test = vec_model.transform(X_test).toarray()

>Convertimos el texto a una matriz de recuento de palabras

>Dado que los datos que tenemos en binario utilizaremos primero Bernoulli

In [125]:
bernoulli = BernoulliNB()
bernoulli.fit(X_train, y_train)
y_pred_bernoulli = bernoulli.predict(X_test)
y_pred_bernoulli

array([1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 0])

In [126]:
bernoulli_accuracy = accuracy_score(y_test, y_pred_bernoulli)
bernoulli_accuracy

0.7877094972067039

#### Otras implementaciones de naive Bayes

In [127]:
multinomial = MultinomialNB()
multinomial.fit(X_train, y_train)
y_pred_multinomial = multinomial.predict(X_test)
y_pred_multinomial

array([1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 0])

In [128]:
multinomial_accuracy = accuracy_score(y_test, y_pred_multinomial)
multinomial_accuracy

0.8324022346368715

In [129]:
gaussian = GaussianNB()
gaussian.fit(X_train, y_train)
y_pred_gaussian = gaussian.predict(X_test)
y_pred_gaussian

array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0,
       1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0])

In [130]:
gaussian_accuracy = accuracy_score(y_test, y_pred_gaussian)
gaussian_accuracy

0.7206703910614525

>Pese a ser todos los datos en binario, vemos que el multinomial nos ofrece una mejor predicción ya que su accuracy es superior a la de los otros, bernoulli sería el segundo mientras gaussian el tercero.

## Optimización del modelo

>Utilizaremos GridSearchCV para optimizar el modelo Multinomial que es el que mejor resultado nos ha dado por defecto

In [131]:
params_multinomial = {
    'alpha' : [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    'force_alpha' : [True, False],
    'fit_prior' : [True, False]
}

In [132]:
grid_multinomial = GridSearchCV(multinomial, params_multinomial, scoring = "accuracy", cv = 10)
grid_multinomial

,estimator,MultinomialNB()
,param_grid,"{'alpha': [0.01, 0.1, ...], 'fit_prior': [True, False], 'force_alpha': [True, False]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1.0


In [133]:
grid_multinomial.fit(X_train, y_train)

grid_multinomial.best_params_

{'alpha': 2.0, 'fit_prior': False, 'force_alpha': True}

In [134]:
grid_multinomial.best_estimator_

,alpha,2.0
,force_alpha,True
,fit_prior,False
,class_prior,None


In [135]:
best_multinomial = grid_multinomial.best_estimator_
y_pred_best_multinomial = best_multinomial.predict(X_test)
grid_accuracy = accuracy_score(y_test, y_pred_best_multinomial)
multinomial_accuracy, grid_accuracy

(0.8324022346368715, 0.8268156424581006)

>Vamos a optimizar también el resto de modelos

>> Bernoulli

In [136]:
params_bernoulli = {
    'alpha' : [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    'force_alpha' : [True, False],
    'fit_prior' : [True, False]
}

In [137]:
grid_bernoulli = GridSearchCV(bernoulli, params_bernoulli, scoring = "accuracy", cv = 10)
grid_bernoulli

,estimator,BernoulliNB()
,param_grid,"{'alpha': [0.01, 0.1, ...], 'fit_prior': [True, False], 'force_alpha': [True, False]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1.0


In [138]:
grid_bernoulli.fit(X_train, y_train)

grid_bernoulli.best_params_

{'alpha': 0.1, 'fit_prior': True, 'force_alpha': True}

In [139]:
grid_bernoulli.best_estimator_

,alpha,0.1
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [140]:
best_bernoulli = grid_bernoulli.best_estimator_
y_pred_best_bernoulli = best_bernoulli.predict(X_test)
grid_accuracy = accuracy_score(y_test, y_pred_best_bernoulli)
bernoulli_accuracy, grid_accuracy

(0.7877094972067039, 0.8268156424581006)

>> No voy a hacer gaussian ya que no sigue esa distribución

## Mejor modelo

>Optimizados, tanto bernoulli como multinomial me dan el mismo resultado, con en torno a un 0.82, pero multinomial sin optimizar me da un mejor resultado. Guardaremos los tres modelos.

>#### Como conclusión, el mejor modelo de los generados en este proyecto ha resultado ser el multinomial sin optimizar

## Guardado del modelo

In [143]:
with open('/workspaces/adamcn10-intro-ml/models/naive-bayes-bernoulli.pkl', 'wb') as file:
    pickle.dump(bernoulli, file)     

with open('/workspaces/adamcn10-intro-ml/models/naive-bayes-multinomial.pkl', 'wb') as file:
    pickle.dump(multinomial, file)     

with open('/workspaces/adamcn10-intro-ml/models/naive-bayes-gaussian.pkl', 'wb') as file:
    pickle.dump(gaussian, file)     

with open('/workspaces/adamcn10-intro-ml/models/naive-bayes-bernoulli-grid.pkl', 'wb') as file:
    pickle.dump(best_bernoulli, file)     

with open('/workspaces/adamcn10-intro-ml/models/naive-bayes-multinomial-grid.pkl', 'wb') as file:
    pickle.dump(best_multinomial, file)     